# Lab 23 — Representation Learning for Time Series with TensorFlow

[Open this lab in Google Colab](https://colab.research.google.com/github/wanghemath/Book-MachineLearning2sda/blob/main/labs/chapter-23-representation-learning-time-series-lab.ipynb)

This lab is designed for independent study. It develops the mathematical ideas first, then turns them into a TensorFlow/Keras workflow for learning embeddings of time-series windows.

## Learning goals

By the end of this lab, you should be able to:

1. Explain what a time-series representation or embedding is.
2. Construct window-level representations by hand.
3. Use PCA as a linear representation method.
4. Train a neural autoencoder for window reconstruction.
5. Extract latent embeddings from a trained encoder.
6. Use embeddings for clustering, retrieval, and anomaly detection.
7. Train a masked-reconstruction model as a self-supervised pretext task.
8. Recognize validation leakage risks in representation-learning pipelines.

We use TensorFlow/Keras because it is available in Google Colab and provides a clear way to define encoders, decoders, and reusable embedding models.


## 1. Mathematical background: representations

A time-series window is a finite block of observations. For a univariate series, a window of length $L$ can be written as

$$
\mathbf x_i = (y_i, y_{i+1}, \ldots, y_{i+L-1})^T \in R^L.
$$

For a multivariate time series with $p$ variables, a window is a matrix

$$
X_i \in R^{L \times p}.
$$

A representation-learning model uses an encoder

$$
f_\theta: R^{L \times p} \to R^d,
$$

where usually $d$ is much smaller than $L p$. The latent vector

$$
\mathbf z_i = f_\theta(X_i)
$$

is called an **embedding**, **latent representation**, or **feature vector**.

A useful representation should preserve information that matters for the downstream task and remove nuisance variation. Downstream tasks may include forecasting, classification, clustering, retrieval, and anomaly detection.


## 2. Autoencoder objective

An autoencoder has two parts:

$$
\mathbf z_i = f_\theta(X_i),
$$

and

$$
\widehat X_i = g_\phi(\mathbf z_i).
$$

The encoder maps the window to a latent vector. The decoder tries to reconstruct the original window. A common training objective is mean squared reconstruction error:

$$
\min_{\theta,\phi}\;\frac1N\sum_{i=1}^N \|X_i - g_\phi(f_\theta(X_i))\|_F^2.
$$

This objective is self-supervised because the training target is the input window itself. No external label is required.

### Important warning

A low reconstruction error does not automatically mean the representation is useful for forecasting, classification, or scientific interpretation. Reconstruction is only one pretext task. Always evaluate representations using a meaningful downstream goal.


## 3. Representation-learning workflow

A careful workflow is:

1. Create windows.
2. Split train/validation/test without leakage.
3. Fit preprocessing only on the training set.
4. Learn or compute representations on the training set.
5. Evaluate representation quality on held-out data.
6. Use visualizations only as diagnostics, not as proof.

In real forecasting projects, overlapping windows from the same time series should be split chronologically. In this synthetic lab, each window is generated independently from a pattern family, so we can use stratified train/test splitting. The leakage principle is still the same: the test set must not influence scaling, fitting, or model selection.


## 4. Programming setup

The next cell imports the packages used in the lab. If TensorFlow is missing outside Colab, install it or run this notebook in Google Colab.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score, confusion_matrix
from sklearn.neighbors import NearestNeighbors

try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "TensorFlow is required for this lab. In Google Colab it is usually preinstalled. "
        "If needed, run: pip install tensorflow"
    ) from exc

np.random.seed(7339)
tf.keras.utils.set_random_seed(7339)

print("TensorFlow version:", tf.__version__)


## 5. Simulate a library of time-series windows

We create a synthetic dataset with four pattern families. This gives us labels for evaluation, but the autoencoder will not use the labels while training.

The four families are:

1. smooth seasonal behavior,
2. trend-dominated behavior,
3. pulse or event behavior,
4. high-frequency oscillatory behavior.

Each window has random phase, amplitude, level, and noise. This forces the representation method to learn meaningful shape information rather than memorize one fixed template.


In [ ]:
def simulate_window_library(n_per_class=260, length=96, seed=7339):
    rng = np.random.default_rng(seed)
    t = np.arange(length)
    windows = []
    labels = []
    names = ["seasonal", "trend", "pulse", "high_freq"]

    for cls in range(4):
        for _ in range(n_per_class):
            level = rng.normal(0.0, 0.3)
            scale = rng.uniform(0.7, 1.4)
            noise = rng.normal(0.0, 0.18, size=length)
            phase = rng.uniform(0, 2 * np.pi)

            if cls == 0:
                x = (
                    level
                    + scale * np.sin(2 * np.pi * t / 24 + phase)
                    + 0.35 * np.cos(2 * np.pi * t / 12 + 0.5 * phase)
                    + noise
                )
            elif cls == 1:
                slope = rng.uniform(-0.035, 0.035)
                centered_t = t - t.mean()
                x = level + slope * centered_t + 0.35 * np.sin(2 * np.pi * t / 48 + phase) + noise
            elif cls == 2:
                center = rng.integers(20, length - 20)
                width = rng.uniform(4, 10)
                pulse = scale * np.exp(-0.5 * ((t - center) / width) ** 2)
                x = level + 0.25 * np.sin(2 * np.pi * t / 24 + phase) + pulse + noise
            else:
                x = (
                    level
                    + 0.55 * scale * np.sin(2 * np.pi * t / 8 + phase)
                    + 0.35 * np.sin(2 * np.pi * t / 6 + 0.7 * phase)
                    + noise
                )

            windows.append(x.astype("float32"))
            labels.append(cls)

    X = np.array(windows, dtype="float32")[:, :, None]
    y = np.array(labels, dtype="int64")
    return X, y, names

X, y, class_names = simulate_window_library()
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Classes:", dict(enumerate(class_names)))


In [ ]:
def plot_example_windows(X, y, class_names, n_examples=4):
    rng = np.random.default_rng(7339)
    t = np.arange(X.shape[1])
    fig, axes = plt.subplots(len(class_names), 1, figsize=(10, 8), sharex=True)
    for cls, ax in enumerate(axes):
        idx = np.where(y == cls)[0]
        chosen = rng.choice(idx, size=n_examples, replace=False)
        for j in chosen:
            ax.plot(t, X[j, :, 0], alpha=0.75)
        ax.set_title(class_names[cls])
        ax.set_ylabel("value")
    axes[-1].set_xlabel("time within window")
    plt.tight_layout()
    plt.show()

plot_example_windows(X, y, class_names)


## 6. Split data and standardize without leakage

The standardization parameters must be learned only from the training set. This matters because representation learning can easily hide data leakage inside preprocessing.


In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=7339, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=7339, stratify=y_temp
)

train_mean = X_train.mean(axis=(0, 1), keepdims=True)
train_std = X_train.std(axis=(0, 1), keepdims=True) + 1e-8

X_train_s = (X_train - train_mean) / train_std
X_val_s = (X_val - train_mean) / train_std
X_test_s = (X_test - train_mean) / train_std

print("Train/val/test shapes:")
print(X_train_s.shape, X_val_s.shape, X_test_s.shape)
print("Training mean after scaling:", float(X_train_s.mean()))
print("Training std after scaling:", float(X_train_s.std()))


## 7. Handcrafted representations

Before training a neural representation, build an interpretable baseline. For each window, we compute:

- mean,
- standard deviation,
- minimum and maximum,
- slope from a linear trend fit,
- lag-1 autocorrelation,
- roughness from first differences,
- low-frequency spectral energy,
- high-frequency spectral energy.

This feature map is already a representation:

$$
\Phi(X_i) \in R^q.
$$


In [ ]:
def window_features(X_windows):
    X2 = X_windows[:, :, 0]
    n, L = X2.shape
    t = np.arange(L)
    centered_t = t - t.mean()
    denom_t = np.sum(centered_t ** 2)

    means = X2.mean(axis=1)
    stds = X2.std(axis=1)
    mins = X2.min(axis=1)
    maxs = X2.max(axis=1)
    slopes = ((X2 - means[:, None]) @ centered_t) / denom_t
    roughness = np.std(np.diff(X2, axis=1), axis=1)

    x0 = X2 - means[:, None]
    lag_num = np.sum(x0[:, 1:] * x0[:, :-1], axis=1)
    lag_den = np.sum(x0 ** 2, axis=1) + 1e-8
    lag1 = lag_num / lag_den

    fft_power = np.abs(np.fft.rfft(x0, axis=1)) ** 2
    low_energy = fft_power[:, 1:6].sum(axis=1)
    high_energy = fft_power[:, 12:].sum(axis=1)
    total_energy = fft_power[:, 1:].sum(axis=1) + 1e-8

    features = np.column_stack([
        means,
        stds,
        mins,
        maxs,
        slopes,
        lag1,
        roughness,
        low_energy / total_energy,
        high_energy / total_energy,
    ])
    names = [
        "mean", "std", "min", "max", "slope", "lag1_acf",
        "roughness", "low_freq_share", "high_freq_share"
    ]
    return features.astype("float32"), names

F_train, feature_names = window_features(X_train_s)
F_test, _ = window_features(X_test_s)

feature_scaler = StandardScaler()
F_train_std = feature_scaler.fit_transform(F_train)
F_test_std = feature_scaler.transform(F_test)

pd.DataFrame(F_train_std, columns=feature_names).head()


In [ ]:
def plot_embedding_2d(Z2, labels, title, class_names):
    plt.figure(figsize=(7, 5))
    for cls, name in enumerate(class_names):
        mask = labels == cls
        plt.scatter(Z2[mask, 0], Z2[mask, 1], s=18, alpha=0.75, label=name)
    plt.xlabel("component 1")
    plt.ylabel("component 2")
    plt.title(title)
    plt.legend()
    plt.grid(alpha=0.25)
    plt.show()

pca_features = PCA(n_components=2, random_state=7339)
Z_feat_train_2d = pca_features.fit_transform(F_train_std)
Z_feat_test_2d = pca_features.transform(F_test_std)

print("Feature PCA explained variance ratio:", np.round(pca_features.explained_variance_ratio_, 3))
plot_embedding_2d(Z_feat_test_2d, y_test, "Handcrafted-feature representation", class_names)


### Interpretation checkpoint

Ask yourself:

1. Which classes are separated well by handcrafted features?
2. Which classes overlap?
3. Does this representation emphasize level, trend, frequency, or local events?

A strong representation-learning pipeline should beat simple feature baselines for the intended task. If it does not, the deep model may be unnecessary.


## 8. PCA as a linear representation

Now we flatten each window and apply PCA directly to the raw window values. PCA gives a linear encoder:

$$
\mathbf z_i = V_d^T(\mathbf x_i - \bar{\mathbf x}).
$$

The reconstruction is

$$
\widehat{\mathbf x}_i = \bar{\mathbf x} + V_d\mathbf z_i.
$$

PCA is useful as a baseline because it is fast, deterministic, and interpretable.


In [ ]:
L = X_train_s.shape[1]
X_train_flat = X_train_s.reshape(len(X_train_s), -1)
X_test_flat = X_test_s.reshape(len(X_test_s), -1)

pca_window = PCA(n_components=8, random_state=7339)
Z_pca_train = pca_window.fit_transform(X_train_flat)
Z_pca_test = pca_window.transform(X_test_flat)
X_test_recon_flat = pca_window.inverse_transform(Z_pca_test)
X_test_recon = X_test_recon_flat.reshape(X_test_s.shape)

print("Cumulative explained variance with 8 components:", round(pca_window.explained_variance_ratio_.sum(), 3))
print("Individual explained variance ratios:", np.round(pca_window.explained_variance_ratio_, 3))

pca_of_pca = PCA(n_components=2, random_state=7339)
Z_pca_test_2d = pca_of_pca.fit_transform(Z_pca_test)
plot_embedding_2d(Z_pca_test_2d, y_test, "PCA representation of raw windows", class_names)


In [ ]:
def plot_reconstruction_examples(X_true, X_recon, labels, class_names, n_examples=4):
    rng = np.random.default_rng(2025)
    chosen = rng.choice(np.arange(len(X_true)), size=n_examples, replace=False)
    t = np.arange(X_true.shape[1])
    fig, axes = plt.subplots(n_examples, 1, figsize=(10, 7), sharex=True)
    for ax, idx in zip(axes, chosen):
        ax.plot(t, X_true[idx, :, 0], label="true")
        ax.plot(t, X_recon[idx, :, 0], linestyle="--", label="reconstruction")
        ax.set_title(f"class: {class_names[labels[idx]]}")
        ax.legend(loc="upper right")
    axes[-1].set_xlabel("time within window")
    plt.tight_layout()
    plt.show()

plot_reconstruction_examples(X_test_s, X_test_recon, y_test, class_names)


## 9. Neural autoencoder representation

PCA is linear. A neural autoencoder can learn nonlinear representations. We use a dense autoencoder first because it is simple and makes the bottleneck explicit.

The architecture is:

$$
X \to \text{Flatten} \to \text{Dense layers} \to \mathbf z \to \text{Dense layers} \to \widehat X.
$$

The layer named `latent` is the embedding layer.


In [ ]:
def build_dense_autoencoder(length, channels=1, latent_dim=8):
    inputs = keras.Input(shape=(length, channels), name="window")
    x = layers.Flatten(name="flatten")(inputs)
    x = layers.Dense(96, activation="relu", name="enc_dense_1")(x)
    x = layers.Dense(48, activation="relu", name="enc_dense_2")(x)
    z = layers.Dense(latent_dim, activation=None, name="latent")(x)

    x = layers.Dense(48, activation="relu", name="dec_dense_1")(z)
    x = layers.Dense(96, activation="relu", name="dec_dense_2")(x)
    x = layers.Dense(length * channels, activation=None, name="reconstruction_flat")(x)
    outputs = layers.Reshape((length, channels), name="reconstruction")(x)

    autoencoder = keras.Model(inputs, outputs, name="dense_autoencoder")
    encoder = keras.Model(inputs, z, name="dense_encoder")
    autoencoder.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss="mse")
    return autoencoder, encoder

autoencoder, encoder = build_dense_autoencoder(length=L, channels=1, latent_dim=8)
autoencoder.summary()


In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=6, restore_best_weights=True
    )
]

history = autoencoder.fit(
    X_train_s,
    X_train_s,
    validation_data=(X_val_s, X_val_s),
    epochs=45,
    batch_size=64,
    callbacks=callbacks,
    verbose=1,
)


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history.history["loss"], label="train")
plt.plot(history.history["val_loss"], label="validation")
plt.xlabel("epoch")
plt.ylabel("MSE reconstruction loss")
plt.title("Autoencoder training curve")
plt.legend()
plt.grid(alpha=0.25)
plt.show()


In [ ]:
X_test_ae_recon = autoencoder.predict(X_test_s, verbose=0)
plot_reconstruction_examples(X_test_s, X_test_ae_recon, y_test, class_names)

recon_mse = np.mean((X_test_s - X_test_ae_recon) ** 2, axis=(1, 2))
print("Test reconstruction MSE summary:")
print(pd.Series(recon_mse).describe())


## 10. Extract and evaluate latent embeddings

The encoder is now a reusable representation model:

$$
X_i \mapsto \mathbf z_i.
$$

We evaluate the embeddings in three ways:

1. two-dimensional visualization,
2. clustering quality,
3. nearest-neighbor retrieval.


In [ ]:
Z_train = encoder.predict(X_train_s, verbose=0)
Z_test = encoder.predict(X_test_s, verbose=0)

print("Latent train shape:", Z_train.shape)
print("Latent test shape:", Z_test.shape)

pca_latent = PCA(n_components=2, random_state=7339)
Z_train_2d = pca_latent.fit_transform(Z_train)
Z_test_2d = pca_latent.transform(Z_test)

plot_embedding_2d(Z_test_2d, y_test, "Neural autoencoder latent representation", class_names)


In [ ]:
kmeans = KMeans(n_clusters=4, n_init=25, random_state=7339)
cluster_test = kmeans.fit_predict(Z_test)

ari = adjusted_rand_score(y_test, cluster_test)
sil = silhouette_score(Z_test, cluster_test)

print("Adjusted Rand Index against true pattern labels:", round(ari, 3))
print("Silhouette score in latent space:", round(sil, 3))
print("Confusion matrix rows=true class, columns=cluster:")
print(confusion_matrix(y_test, cluster_test))


### Interpretation checkpoint

Clustering metrics are diagnostic tools, not final proof. If the classes overlap, possible explanations include:

- the simulated classes are genuinely similar,
- the bottleneck is too small,
- the autoencoder objective reconstructs nuisance variation,
- the model has not trained enough,
- the input standardization removed important information.


## 11. Retrieval in latent space

Embedding spaces are often used for retrieval. Given a query window, we find its nearest neighbors by Euclidean distance in latent space.

Good retrieval means that nearest neighbors share meaningful structure with the query.


In [ ]:
nbrs = NearestNeighbors(n_neighbors=5)
nbrs.fit(Z_train)

query_index = 7
distances, neighbor_indices = nbrs.kneighbors(Z_test[query_index:query_index + 1])
neighbor_indices = neighbor_indices[0]
distances = distances[0]

t = np.arange(L)
fig, axes = plt.subplots(1 + len(neighbor_indices), 1, figsize=(10, 9), sharex=True)
axes[0].plot(t, X_test_s[query_index, :, 0])
axes[0].set_title(f"Query: true class {class_names[y_test[query_index]]}")

for ax, idx, dist in zip(axes[1:], neighbor_indices, distances):
    ax.plot(t, X_train_s[idx, :, 0])
    ax.set_title(f"Neighbor: class {class_names[y_train[idx]]}, latent distance={dist:.3f}")

axes[-1].set_xlabel("time within window")
plt.tight_layout()
plt.show()


## 12. Anomaly detection with reconstruction error

Autoencoders are often used for anomaly detection. The idea is:

1. Train on normal patterns.
2. Reconstruct new windows.
3. Flag windows with large reconstruction error.

This works only when anomalies are hard for the model to reconstruct. It can fail if the autoencoder is too flexible.


In [ ]:
def simulate_anomaly_windows(n=180, length=96, seed=9001):
    rng = np.random.default_rng(seed)
    t = np.arange(length)
    anomalies = []
    kinds = []
    for i in range(n):
        base = 0.5 * np.sin(2 * np.pi * t / 24 + rng.uniform(0, 2 * np.pi))
        noise = rng.normal(0, 0.18, size=length)
        x = base + noise
        kind = i % 3
        if kind == 0:
            # extreme spike or drop
            center = rng.integers(15, length - 15)
            x[center:center + 3] += rng.choice([-1, 1]) * rng.uniform(4.0, 5.5)
            kinds.append("spike")
        elif kind == 1:
            # abrupt level shift
            cut = rng.integers(25, length - 25)
            x[cut:] += rng.choice([-1, 1]) * rng.uniform(2.5, 3.5)
            kinds.append("level_shift")
        else:
            # unusual square-wave pattern
            x += 1.8 * np.sign(np.sin(2 * np.pi * t / 10 + rng.uniform(0, 2 * np.pi)))
            kinds.append("square_wave")
        anomalies.append(x.astype("float32"))
    return np.array(anomalies, dtype="float32")[:, :, None], np.array(kinds)

X_anom, anomaly_kind = simulate_anomaly_windows(length=L)
X_anom_s = (X_anom - train_mean) / train_std

X_train_recon = autoencoder.predict(X_train_s, verbose=0)
X_test_recon = autoencoder.predict(X_test_s, verbose=0)
X_anom_recon = autoencoder.predict(X_anom_s, verbose=0)

train_error = np.mean((X_train_s - X_train_recon) ** 2, axis=(1, 2))
test_error = np.mean((X_test_s - X_test_recon) ** 2, axis=(1, 2))
anom_error = np.mean((X_anom_s - X_anom_recon) ** 2, axis=(1, 2))

threshold = np.quantile(train_error, 0.95)
false_positive_rate = np.mean(test_error > threshold)
detection_rate = np.mean(anom_error > threshold)

print("95% training threshold:", round(float(threshold), 4))
print("False positive rate on normal test windows:", round(float(false_positive_rate), 3))
print("Detection rate on anomaly windows:", round(float(detection_rate), 3))


In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(test_error, bins=30, alpha=0.7, label="normal test")
plt.hist(anom_error, bins=30, alpha=0.7, label="anomalies")
plt.axvline(threshold, linestyle="--", label="95% train threshold")
plt.xlabel("reconstruction MSE")
plt.ylabel("count")
plt.title("Autoencoder reconstruction-error anomaly score")
plt.legend()
plt.grid(alpha=0.25)
plt.show()


In [ ]:
# Show several anomalous windows and their reconstructions.
rng = np.random.default_rng(42)
chosen = rng.choice(np.arange(len(X_anom_s)), size=4, replace=False)
fig, axes = plt.subplots(len(chosen), 1, figsize=(10, 7), sharex=True)
for ax, idx in zip(axes, chosen):
    ax.plot(t, X_anom_s[idx, :, 0], label="anomaly")
    ax.plot(t, X_anom_recon[idx, :, 0], linestyle="--", label="reconstruction")
    ax.set_title(f"{anomaly_kind[idx]}, error={anom_error[idx]:.3f}")
    ax.legend(loc="upper right")
axes[-1].set_xlabel("time within window")
plt.tight_layout()
plt.show()


## 13. Masked reconstruction as a self-supervised pretext task

A masked-reconstruction model receives a corrupted window and learns to reconstruct the original window. This is similar in spirit to masked modeling for language and sequence data.

Let $M_i$ be a binary mask. The corrupted input is

$$
\widetilde X_i = M_i \odot X_i.
$$

The model is trained to predict $X_i$ from $\widetilde X_i$.

This objective encourages the encoder to use context rather than simply copying each input coordinate.


In [ ]:
def random_block_mask(X_windows, mask_fraction=0.25, block_length=8, seed=7339):
    rng = np.random.default_rng(seed)
    X_masked = X_windows.copy()
    n, L, c = X_masked.shape
    n_blocks = max(1, int(mask_fraction * L / block_length))
    for i in range(n):
        for _ in range(n_blocks):
            start = rng.integers(0, L - block_length + 1)
            X_masked[i, start:start + block_length, :] = 0.0
    return X_masked

X_train_masked = random_block_mask(X_train_s, seed=1)
X_val_masked = random_block_mask(X_val_s, seed=2)
X_test_masked = random_block_mask(X_test_s, seed=3)

fig, ax = plt.subplots(figsize=(10, 4))
example = 0
ax.plot(t, X_train_s[example, :, 0], label="original")
ax.plot(t, X_train_masked[example, :, 0], linestyle="--", label="masked input")
ax.set_title("Example of block masking")
ax.legend()
plt.show()


In [ ]:
masked_autoencoder, masked_encoder = build_dense_autoencoder(length=L, channels=1, latent_dim=8)

masked_history = masked_autoencoder.fit(
    X_train_masked,
    X_train_s,
    validation_data=(X_val_masked, X_val_s),
    epochs=45,
    batch_size=64,
    callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True)],
    verbose=1,
)


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(masked_history.history["loss"], label="train")
plt.plot(masked_history.history["val_loss"], label="validation")
plt.xlabel("epoch")
plt.ylabel("masked reconstruction loss")
plt.title("Masked-reconstruction training curve")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

X_masked_recon = masked_autoencoder.predict(X_test_masked, verbose=0)
plot_reconstruction_examples(X_test_s, X_masked_recon, y_test, class_names)


## 14. Compare representation spaces

We now compare three representations on the same test set:

1. handcrafted features,
2. PCA of raw windows,
3. neural autoencoder embeddings,
4. masked-reconstruction embeddings.

There is no universally best representation. The right representation depends on the downstream objective.


In [ ]:
Z_masked_test = masked_encoder.predict(X_test_s, verbose=0)
Z_masked_2d = PCA(n_components=2, random_state=7339).fit_transform(Z_masked_test)

plot_embedding_2d(Z_feat_test_2d, y_test, "Handcrafted features", class_names)
plot_embedding_2d(Z_pca_test_2d, y_test, "PCA features", class_names)
plot_embedding_2d(Z_test_2d, y_test, "Autoencoder latent features", class_names)
plot_embedding_2d(Z_masked_2d, y_test, "Masked-reconstruction latent features", class_names)


In [ ]:
def clustering_report(Z, labels, name):
    pred = KMeans(n_clusters=4, n_init=25, random_state=7339).fit_predict(Z)
    ari = adjusted_rand_score(labels, pred)
    sil = silhouette_score(Z, pred)
    return {"representation": name, "adjusted_rand": ari, "silhouette": sil}

reports = [
    clustering_report(F_test_std, y_test, "handcrafted"),
    clustering_report(Z_pca_test, y_test, "pca_window"),
    clustering_report(Z_test, y_test, "autoencoder"),
    clustering_report(Z_masked_test, y_test, "masked_autoencoder"),
]

pd.DataFrame(reports).sort_values("adjusted_rand", ascending=False)


## 15. Optional extension: contrastive thinking without a heavy training loop

Contrastive representation learning asks the encoder to place two augmented views of the same window close together and views from different windows farther apart.

Typical time-series augmentations include:

- jittering with small noise,
- small amplitude scaling,
- mild time masking,
- small shifts.

But augmentations are assumptions. For example, time shifting may be harmless for periodic vibration signals but harmful when event timing is clinically or economically meaningful.


In [ ]:
def augment_windows(X_windows, seed=123):
    rng = np.random.default_rng(seed)
    X_aug = X_windows.copy()
    n, L, c = X_aug.shape

    # Small amplitude scaling.
    scales = rng.normal(loc=1.0, scale=0.05, size=(n, 1, 1)).astype("float32")
    X_aug = X_aug * scales

    # Small jitter.
    X_aug = X_aug + rng.normal(0.0, 0.05, size=X_aug.shape).astype("float32")

    # Mild random shift.
    for i in range(n):
        shift = rng.integers(-3, 4)
        X_aug[i, :, 0] = np.roll(X_aug[i, :, 0], shift)

    return X_aug.astype("float32")

X_view1 = augment_windows(X_test_s, seed=10)
X_view2 = augment_windows(X_test_s, seed=11)

Z_view1 = encoder.predict(X_view1, verbose=0)
Z_view2 = encoder.predict(X_view2, verbose=0)

positive_dist = np.linalg.norm(Z_view1 - Z_view2, axis=1)
negative_dist = np.linalg.norm(Z_view1 - np.roll(Z_view2, shift=1, axis=0), axis=1)

print("Mean positive-pair distance:", round(float(positive_dist.mean()), 3))
print("Mean negative-pair distance:", round(float(negative_dist.mean()), 3))

plt.figure(figsize=(7, 4))
plt.hist(positive_dist, bins=25, alpha=0.7, label="same window, two views")
plt.hist(negative_dist, bins=25, alpha=0.7, label="different windows")
plt.xlabel("latent distance")
plt.ylabel("count")
plt.title("A simple contrastive diagnostic")
plt.legend()
plt.grid(alpha=0.25)
plt.show()


## 16. Exercises

1. Change the latent dimension from 8 to 2, 4, 16, and 32. How do reconstruction quality and clustering quality change?
2. Replace the dense autoencoder with a Conv1D autoencoder. Does it improve shape-based clustering?
3. Train the autoencoder only on two classes. Are the other two classes easier to detect as anomalies?
4. Create a chronological splitting experiment from one long simulated series. What changes compared with independent-window splitting?
5. Modify the anomaly generator so that anomalies are subtle instead of extreme. Does reconstruction error still work?
6. Compare global standardization with per-window standardization. What information does each preprocessing choice remove?


## 17. Mini-project: representation audit

Choose one representation from this lab and write a short audit report answering:

1. What information does the representation appear to preserve?
2. What information does it appear to discard?
3. What downstream task is it good for?
4. What downstream task might it fail on?
5. Where could leakage enter the pipeline?
6. What simple baseline should it be compared against?

A strong representation-learning report should include both quantitative evidence and plots of representative successes and failures.


## 18. AI-assisted study prompts

Use these prompts to study, not to replace your own analysis.

1. "Explain why reconstruction loss alone may not imply a good forecasting representation. Give a time-series example."
2. "Compare PCA, autoencoder, and masked autoencoder representations for time-series windows. What assumptions does each make?"
3. "Given a time-series representation pipeline, list possible sources of data leakage."
4. "Design a contrastive augmentation strategy for hourly electricity demand. Which augmentations are safe, and which may change the meaning?"
5. "Explain how nearest-neighbor retrieval in latent space can be used to diagnose representation quality."


## 19. Checklist

Before using a learned time-series representation in a project, check:

- [ ] Was preprocessing fit only on training data?
- [ ] Were overlapping windows split without leakage?
- [ ] Was a simple handcrafted or PCA baseline evaluated?
- [ ] Does the latent space preserve task-relevant structure?
- [ ] Are anomalies or rare events handled appropriately?
- [ ] Are visualizations supported by quantitative metrics?
- [ ] Does the representation still work on held-out future data?
